<a href="https://colab.research.google.com/github/makdatascience/Covid19_x-ray_cloudx/blob/main/Chest_X_ray_(Covid_19_%26_Pneumonia)_mod.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
chest_xray_covid19_pneumonia_path = kagglehub.dataset_download('prashant268/chest-xray-covid19-pneumonia')

print('Data source import complete.')


Data source import complete.


In [3]:
# list the files in chest_xray_covid19_pneumonia_path

import os
for dirname, _, filenames in os.walk(chest_xray_covid19_pneumonia_path):
    for filename in filenames:
        print(os.path.join(dirname, filename))


Streaming output truncated to the last 5000 lines.
/kaggle/input/chest-xray-covid19-pneumonia/Data/train/PNEUMONIA/PNEUMONIA(2441).jpg
/kaggle/input/chest-xray-covid19-pneumonia/Data/train/PNEUMONIA/PNEUMONIA(2643).jpg
/kaggle/input/chest-xray-covid19-pneumonia/Data/train/PNEUMONIA/PNEUMONIA(477).jpg
/kaggle/input/chest-xray-covid19-pneumonia/Data/train/PNEUMONIA/PNEUMONIA(1513).jpg
/kaggle/input/chest-xray-covid19-pneumonia/Data/train/PNEUMONIA/PNEUMONIA(611).jpg
/kaggle/input/chest-xray-covid19-pneumonia/Data/train/PNEUMONIA/PNEUMONIA(2916).jpg
/kaggle/input/chest-xray-covid19-pneumonia/Data/train/PNEUMONIA/PNEUMONIA(1729).jpg
/kaggle/input/chest-xray-covid19-pneumonia/Data/train/PNEUMONIA/PNEUMONIA(2139).jpg
/kaggle/input/chest-xray-covid19-pneumonia/Data/train/PNEUMONIA/PNEUMONIA(1697).jpg
/kaggle/input/chest-xray-covid19-pneumonia/Data/train/PNEUMONIA/PNEUMONIA(1501).jpg
/kaggle/input/chest-xray-covid19-pneumonia/Data/train/PNEUMONIA/PNEUMONIA(89).jpg
/kaggle/input/chest-xray-covi

Kaggle Notebook : https://www.kaggle.com/code/mayankbelwal/chest-x-ray-covid-19-pneumonia-mod


In [ ]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
# !pip install matplotlib-venn
# !apt-get -qq install -y libfluidsynth1
# !apt-get -qq install -y libarchive-dev && pip install -U libarchive
# import libarchive
# !apt-get -qq install -y graphviz && pip install pydot
# import pydot
# !pip install cartopy
# import cartopy

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dropout, Flatten, Dense, GlobalAveragePooling2D
from tensorflow.keras.applications import EfficientNetB3  # Use EfficientNetB3 instead of ResNet50
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report, confusion_matrix

# Set dataset paths
my_data_dir1 = r'/kaggle/input/chest-xray-covid19-pneumonia/Data/'
my_data_dir= chest_xray_covid19_pneumonia_path
train_path = my_data_dir + '/Data/train/'
test_path = my_data_dir + '/Data/test/'

# Data augmentation (improved)
image_gen = ImageDataGenerator(
    rotation_range=30,
    width_shift_range=0.15,
    height_shift_range=0.15,
    brightness_range=[0.7, 1.3],  # Improved augmentation
    rescale=1/255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Load EfficientNetB3 (better than ResNet50 for medical images)
base_model = EfficientNetB3(weights="imagenet", include_top=False, input_shape=(224, 224, 3))

# Unfreeze last few layers for fine-tuning
for layer in base_model.layers[-10:]:
    layer.trainable = True

# Build the model
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(3, activation='softmax')  # 3 classes
])

# Compile the model with a smaller learning rate
model.compile(loss='categorical_crossentropy',
              optimizer=Adam(learning_rate=0.0001),
              metrics=['accuracy'])

# Print model summary
model.summary()

# Callbacks (Early stopping + Learning rate reduction)
early_stop = EarlyStopping(monitor='val_loss', patience=3, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, verbose=1)

# Load training and testing data
batch_size = 64

train_image_gen = image_gen.flow_from_directory(train_path,
                                                target_size=(224, 224),
                                                batch_size=batch_size,
                                                class_mode='categorical')

test_image_gen = image_gen.flow_from_directory(test_path,
                                               target_size=(224, 224),
                                               batch_size=batch_size,
                                               class_mode='categorical',
                                               shuffle=False)

# Train the model
history = model.fit(train_image_gen, epochs=20,
                    validation_data=test_image_gen,
                    callbacks=[early_stop, reduce_lr])

# Plot accuracy & loss
loss_df = pd.DataFrame(history.history)
loss_df[['accuracy', 'val_accuracy']].plot()
loss_df[['loss', 'val_loss']].plot()

# Evaluate model
model.evaluate(test_image_gen)

# Predictions
predictions = np.argmax(model.predict(test_image_gen), axis=-1)

# Classification report & confusion matrix
print(classification_report(test_image_gen.classes, predictions))
cm = confusion_matrix(test_image_gen.classes, predictions)
sns.heatmap(cm, annot=True, fmt='d', cmap="Blues")
plt.show()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb3 (Functional)     │ (None, 7, 7, 1536)     │    10,783,535 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1536)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │       393,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │           771 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,177,778 (42.64 MB)

 Trainable params: 11,090,475 (42.31 MB)

 Non-trainable params: 87,303 (341.03 KB)

Found 5144 images belonging to 3 classes.
Found 1288 images belonging to 3 classes.
Epoch 1/20
57/81 ━━━━━━━━━━━━━━━━━━━━ 38s 2s/step - accuracy: 0.6954 - loss: 0.6693